# Kairos OS · Notebook 3: Continual Multi-Task Fine-Tuning

Loads pre-trained **CIDA** + **PlanProjector** weights and performs **continual multi-task fine-tuning** on a mixed corpus (Math + Code + General Chat).

This prevents the "alignment tax" on coding/general tasks while retaining logical math reasoning capabilities.

**Backbone LLM**: Frozen fp16 (Qwen1.5-1.8B-Chat or 7B-Chat)

In [ ]:
!pip install -q torch transformers datasets accelerate torchdiffeq
import subprocess, sys, os
repo = 'CIDA-plugin'
if not os.path.exists(repo):
    subprocess.run(['git','clone','--depth','1',
                   'https://github.com/Kairatzh/CIDA-plugin.git'], check=True)
sys.path.insert(0, os.path.abspath(repo))
from google.colab import drive
drive.mount('/content/drive')
print("Setup complete.")


## 1. Load Frozen LLM & Pre-Trained Weights

In [ ]:
import gc, re, json, time, random, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from tqdm.auto import tqdm
from cida_plugin import CIDAPlugin, CIDAPluginConfig

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WEIGHTS_PATH = '/content/drive/MyDrive/Kairos_CIDA_7B/best'
SAVE_PATH = '/content/drive/MyDrive/Kairos_CIDA_7B/continual_best'

def flush():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ── Load config & metadata ──
info_path = Path(WEIGHTS_PATH) / "train_info.json"
kcfg_path = Path(WEIGHTS_PATH) / "kairos_config.json"
if info_path.exists():
    with open(info_path) as f: info = json.load(f)
elif kcfg_path.exists():
    with open(kcfg_path) as f: info = json.load(f)
else:
    info = {}

MODEL_NAME = info.get("llm_model_name", "Qwen/Qwen1.5-7B-Chat")
NUM_PLAN_TOKENS = info.get("num_plan_tokens", 8)
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

llm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, trust_remote_code=True, torch_dtype=torch.float16,
    device_map="auto", output_hidden_states=True)
for p in llm.parameters(): p.requires_grad = False
llm.eval()

LLM_HIDDEN = llm.config.hidden_size
embed_tokens = llm.get_input_embeddings()
for p in embed_tokens.parameters(): p.requires_grad = False

# ── Re-initialize & Load CIDA weights ──
cida_cfg = CIDAPluginConfig.from_pretrained(WEIGHTS_PATH)
cida = CIDAPlugin(cida_cfg)
sd = torch.load(str(Path(WEIGHTS_PATH) / "pytorch_model.bin"), map_location="cpu")
cida.load_state_dict(sd)
cida = cida.to(DEVICE).float()

# ── PlanProjector ──
class CIDAPlanProjector(nn.Module):
    def __init__(self, d_cida, d_llm, num_agents, num_plan_tokens=8):
        super().__init__()
        self.num_plan_tokens = num_plan_tokens
        self.d_llm = d_llm
        d_flat = num_agents * d_cida
        self.projector = nn.Sequential(
            nn.Linear(d_flat, d_flat * 2), nn.GELU(), nn.Dropout(0.05),
            nn.Linear(d_flat * 2, num_plan_tokens * d_llm),
            nn.LayerNorm(num_plan_tokens * d_llm),
        )
    def forward(self, state_s):
        B = state_s.size(0)
        flat = state_s.reshape(B, -1).float()
        return self.projector(flat).view(B, self.num_plan_tokens, self.d_llm)

plan_projector = CIDAPlanProjector(
    d_cida=cida_cfg.d_hidden, d_llm=LLM_HIDDEN,
    num_agents=cida_cfg.num_agents, num_plan_tokens=NUM_PLAN_TOKENS,
).to(DEVICE)
plan_projector.load_state_dict(
    torch.load(str(Path(WEIGHTS_PATH) / "plan_projector.pt"), map_location=DEVICE)
)

print("Pre-trained model loaded. Only CIDA + PlanProjector will receive gradients.")


## 2. Load & Merge Multi-Task Datasets (Math + Code + General Chat)

We mix:
- **GSM8K**: Math Reasoning
- **MBPP**: Python coding tasks
- **Alpaca**: General chat / instructions

In [ ]:
print("Loading datasets...")
gsm = list(load_dataset("openai/gsm8k", "main", split="train"))
mbpp = list(load_dataset("google-research-datasets/mbpp", split="train"))
alpaca = list(load_dataset("tatsu-lab/alpaca", split="train"))

# Mix tasks into a unified record list
records = []

# 1. Math Tasks
for ex in gsm:
    prompt = f"<|im_start|>system\nYou are a mathematical reasoner. Solve step by step, then state the final answer after '####'.\n<|im_end|>\n<|im_start|>user\n{ex['question']}\n<|im_end|>\n<|im_start|>assistant\n"
    records.append({"prompt": prompt, "answer": ex["answer"], "task": "math"})

# 2. Coding Tasks
for ex in mbpp:
    # Use code block format
    prompt = f"<|im_start|>system\nYou are a Python programming expert. Write clean, correct code inside a python markdown block.\n<|im_end|>\n<|im_start|>user\nWrite a Python function for: {ex['text']}\n<|im_end|>\n<|im_start|>assistant\n"
    # Ensure answer has markdown block wrapper
    ans = f"```python\n{ex['code']}\n```"
    records.append({"prompt": prompt, "answer": ans, "task": "code"})

# 3. General Chat
for ex in alpaca[:2500]: # take a subset of Alpaca to balance the training size
    prompt = f"<|im_start|>system\nYou are a helpful assistant.\n<|im_end|>\n<|im_start|>user\n{ex['instruction']}\n{ex['input']}\n<|im_end|>\n<|im_start|>assistant\n"
    records.append({"prompt": prompt, "answer": ex["output"], "task": "chat"})

random.seed(42)
random.shuffle(records)
print(f"Total merged records: {len(records)}")
print(f"  Math  : {len(gsm)}")
print(f"  Code  : {len(mbpp)}")
print(f"  Chat  : {min(2500, len(alpaca))}")

# Split into Train / Val
split_idx = int(0.9 * len(records))
train_records = records[:split_idx]
val_records = records[split_idx:]
print(f"Train size: {len(train_records)} | Val size: {len(val_records)}")


In [ ]:
class MultiTaskDataset(Dataset):
    def __init__(self, records, max_prompt=256, max_answer=128):
        self.records = records
        self.max_prompt = max_prompt
        self.max_answer = max_answer

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        tokenizer.padding_side = "left"
        p_enc = tokenizer(rec["prompt"], max_length=self.max_prompt, truncation=True,
                          padding="max_length", return_tensors="pt")
        tokenizer.padding_side = "right"
        a_enc = tokenizer(rec["answer"], max_length=self.max_answer, truncation=True,
                          padding="max_length", return_tensors="pt")
        tokenizer.padding_side = "left"
        return {
            "prompt_ids": p_enc["input_ids"].squeeze(0),
            "prompt_mask": p_enc["attention_mask"].squeeze(0),
            "answer_ids": a_enc["input_ids"].squeeze(0),
            "answer_mask": a_enc["attention_mask"].squeeze(0),
            "task": rec["task"]
        }

train_ds = MultiTaskDataset(train_records)
val_ds = MultiTaskDataset(val_records)
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=2, shuffle=False, num_workers=2, pin_memory=True)
print("DataLoader shapes verified.")


## 3. Forward Pass & Loss with Causal Shift

In [ ]:
@torch.no_grad()
def get_prompt_context(prompt_ids, prompt_mask):
    out = llm(input_ids=prompt_ids, attention_mask=prompt_mask,
              output_hidden_states=True, use_cache=False)
    last_hidden = out.hidden_states[-1]
    seq_len = prompt_mask.sum(dim=1) - 1
    batch_idx = torch.arange(prompt_ids.size(0), device=prompt_ids.device)
    pooled = last_hidden[batch_idx, seq_len].float()
    prompt_embeds = embed_tokens(prompt_ids.to(embed_tokens.weight.device)).to(DEVICE).detach().float()
    return pooled, prompt_embeds

def cida_forward_with_loss(prompt_ids, prompt_mask, answer_ids, answer_mask):
    B, Lq = prompt_ids.shape
    La = answer_ids.shape[1]
    P = NUM_PLAN_TOKENS
    llm_dtype = next(llm.parameters()).dtype

    pooled, prompt_embeds = get_prompt_context(prompt_ids, prompt_mask)
    cida_out = cida(pooled=pooled)
    plan_embeds = plan_projector(cida_out["state_final"].s)

    with torch.no_grad():
        answer_embeds = embed_tokens(answer_ids.to(embed_tokens.weight.device)).to(DEVICE).float()

    train_embeds = torch.cat([prompt_embeds, plan_embeds, answer_embeds], dim=1)
    plan_mask = torch.ones(B, P, dtype=prompt_mask.dtype, device=DEVICE)
    combined_mask = torch.cat([prompt_mask, plan_mask, answer_mask], dim=1)

    with torch.cuda.amp.autocast():
        lm_out = llm(inputs_embeds=train_embeds.to(llm_dtype), attention_mask=combined_mask, use_cache=False)
    logits = lm_out.logits.float()

    answer_start = Lq + P
    answer_logits = logits[:, answer_start - 1 : answer_start + La - 1, :]
    answer_targets = answer_ids
    loss_mask = answer_mask.bool()

    B_flat = B * La
    logits_f = answer_logits.reshape(B_flat, -1)
    targets_f = answer_targets.reshape(B_flat).masked_fill(~loss_mask.reshape(B_flat), 0)
    mask_f = loss_mask.reshape(B_flat)

    loss_per_tok = F.cross_entropy(logits_f, targets_f, reduction="none")
    loss = (loss_per_tok * mask_f.float()).sum() / mask_f.float().sum().clamp(min=1)
    return loss, cida_out


## 4. Fine-Tuning Optimizer & Scheduler

In [ ]:
trainable_params = list(cida.parameters()) + list(plan_projector.parameters())
optimizer = torch.optim.AdamW(trainable_params, lr=1e-4, weight_decay=1e-4, betas=(0.9, 0.98))

EPOCHS = 1
GRAD_ACCUM = 8
total_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-6)
scaler = torch.cuda.amp.GradScaler(enabled=True)

print(f"Fine-tuning starting: {total_steps} optimizer steps.")


In [ ]:
cida.train()
plan_projector.train()
global_step = 0

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    optimizer.zero_grad(set_to_none=True)
    
    for batch_idx, batch in enumerate(train_loader):
        prompt_ids  = batch["prompt_ids"].to(DEVICE)
        prompt_mask = batch["prompt_mask"].to(DEVICE)
        answer_ids  = batch["answer_ids"].to(DEVICE)
        answer_mask = batch["answer_mask"].to(DEVICE)

        with torch.cuda.amp.autocast(enabled=True):
            loss, cida_out = cida_forward_with_loss(prompt_ids, prompt_mask, answer_ids, answer_mask)
            loss = loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        epoch_loss += loss.item() * GRAD_ACCUM

        if (batch_idx + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

            if global_step % 20 == 0:
                print(f"Step {global_step:>4} / {total_steps} | Loss: {epoch_loss / (batch_idx+1):.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")

# Save the final checkpoint to Google Drive
save_dir = Path(SAVE_PATH)
save_dir.mkdir(parents=True, exist_ok=True)
cida.save_pretrained(str(save_dir))
torch.save(plan_projector.state_dict(), str(save_dir / "plan_projector.pt"))

info_train = {
    "llm_model_name": MODEL_NAME,
    "num_plan_tokens": NUM_PLAN_TOKENS,
    "cida_d_hidden": cida_cfg.d_hidden,
    "llm_hidden": LLM_HIDDEN,
    "info": "Continual multi-task fine-tuning"
}
with open(str(save_dir / "kairos_config.json"), "w") as f:
    json.dump(info_train, f, indent=2)

print(f"Continual fine-tuning complete! Weights saved to: {SAVE_PATH}")
